# 23 - Pengujian Performa Sistem QA RAG Lokal

1. **Produksi**: `/api/ask`, RAG wajib, cache aplikasi aktif sesuai konfigurasi.
2. **Evaluasi privat**: `/api/eval/ask`, RAG/non-RAG, cache aplikasi dilewati, konteks asli tersedia.
3. **Streaming web**: `/api/ask/stream`, ukur token draf pertama, jawaban setelah pemeriksaan, dan selesai seluruh proses.
4. **Generator terpisah**: Ollama `/api/generate`, token/detik dan resource. Ini bukan payload/prompt `/api/chat` aplikasi.

Fokus pengujian: waktu pencarian dokumen, generasi jawaban, respons keseluruhan, dan memori.
Pengukuran RAM khusus backend belum tersedia; RAM Ollama dan RAM seluruh komputer dicatat terpisah.
Jika kernel meminta restart setelah pemasangan dependensi, restart lalu jalankan ulang dari sel lokasi.

In [18]:
# Lokasi tetap: notebook boleh dijalankan dari folder mana pun di komputer ini.
import os
from pathlib import Path

PROJECT_ROOT = Path(r"D:/Politeknik Statistika STIS/Pembelajaran/Skripsi/SourceCode/Sistem QA RAG")
RESULTS_ROOT = Path(r"D:/Politeknik Statistika STIS/Pembelajaran/Skripsi/SourceCode/SkripsiNabil/Hasil/tujuan3")
NOTEBOOK_REQUIREMENTS = PROJECT_ROOT / "requirements-notebook.txt"

if not (PROJECT_ROOT / "app" / "main.py").is_file():
    raise FileNotFoundError(f"Folder sistem QA tidak ditemukan: {PROJECT_ROOT}")
if not NOTEBOOK_REQUIREMENTS.is_file():
    raise FileNotFoundError(f"File dependensi tidak ditemukan: {NOTEBOOK_REQUIREMENTS}")

os.environ["QA_PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["SYSTEM_EVAL_OUTPUT"] = str(RESULTS_ROOT)
print("Folder sistem:", PROJECT_ROOT)
print("Folder hasil:", RESULTS_ROOT)


Folder sistem: D:\Politeknik Statistika STIS\Pembelajaran\Skripsi\SourceCode\Sistem QA RAG
Folder hasil: D:\Politeknik Statistika STIS\Pembelajaran\Skripsi\SourceCode\SkripsiNabil\Hasil\tujuan3


In [19]:
# Memasang ke kernel notebook dari file dependensi di folder sistem.
get_ipython().run_line_magic("pip", f'install -r "{NOTEBOOK_REQUIREMENTS}"')

import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "requests": "requests",
    "psutil": "psutil",
}
missing_packages = [
    package for module, package in required_packages.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    raise RuntimeError(
        "Dependensi notebook belum lengkap: " + ", ".join(missing_packages)
        + ". Instal requirements-notebook.txt pada environment kernel."
    )

# NVML bersifat opsional. Notebook tetap berjalan tanpa paket ini; kolom GPU
# akan kosong dan inventaris mencoba fallback Windows.
if importlib.util.find_spec("pynvml") is None:
    print(
        "INFO: pynvml belum tersedia. Untuk sampling GPU NVIDIA, instal "
        "'nvidia-ml-py'. Benchmark CPU/RAM tetap dapat dijalankan."
    )

import json
import os
import platform
import threading
import time
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import requests

from datetime import datetime
import hashlib
import uuid

root_override = os.environ.get("QA_PROJECT_ROOT")
if root_override:
    PROJECT_ROOT = Path(root_override).resolve()
else:
    PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                         if (p / "app" / "main.py").is_file() and (p / "web").is_dir()), None)
if PROJECT_ROOT is None or not (PROJECT_ROOT / "app" / "main.py").is_file():
    raise RuntimeError("Buka notebook dari folder proyek, atau set QA_PROJECT_ROOT.")


Note: you may need to restart the kernel to use updated packages.
INFO: pynvml belum tersedia. Untuk sampling GPU NVIDIA, instal 'nvidia-ml-py'. Benchmark CPU/RAM tetap dapat dijalankan.


## 1. Konfigurasi sistem lokal

In [20]:
BASE_URL = os.environ.get("RAG_API_URL", "http://127.0.0.1:8000").rstrip("/")
OLLAMA_URL = os.environ.get("INFERENCE_URL", "http://127.0.0.1:11434").rstrip("/")
OUTPUT_BASE = Path(os.environ.get("SYSTEM_EVAL_OUTPUT",
                                      str(PROJECT_ROOT / "artifacts" / "system_performance")))
RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:8]
OUT = OUTPUT_BASE / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

E2E_MODE = "production"  # "production" atau "evaluation"
E2E_RAG_MODES = [True]  # [True, False] hanya pada mode evaluation
if E2E_MODE not in {"production", "evaluation"}:
    raise ValueError("E2E_MODE tidak dikenal.")
if not E2E_RAG_MODES or any(type(v) is not bool for v in E2E_RAG_MODES):
    raise ValueError("E2E_RAG_MODES harus berisi boolean.")
if E2E_MODE == "production" and E2E_RAG_MODES != [True]:
    raise ValueError("Produksi selalu RAG. Gunakan evaluation untuk non-RAG.")
if E2E_MODE == "evaluation" and len(os.environ.get("EVALUATION_API_KEY", "")) < 32:
    raise ValueError("Isi EVALUATION_API_KEY melalui environment kernel.")

# Pilih model yang akan diuji; preflight memeriksa ketersediaan aktual.
MODEL_KEY_TO_OLLAMA_TAG = {
    "llama3.2-finetuned": "llama3.2-finetuned:latest",
    "gemma2-finetuned": "gemma2-finetuned:latest",
}

HARDWARE_NOTES = ""  # Opsional: mode daya, batas TDP, aplikasi lain, atau konfigurasi khusus.

RUN_API_PROBE = True
RUN_FUNCTIONAL_TESTS = True
RUN_STREAM_TEST = True
RUN_E2E_BENCHMARK = True
RUN_GENERATOR_BENCHMARK = True

REQUEST_TIMEOUT = (10, 900)
SAMPLE_INTERVAL_S = 0.05
TOP_K = 5
RESPONSE_STYLE = "detail"

N_E2E_WARMUP = 1
N_E2E_REPEAT = 3       # Tiga ulangan untuk setiap pertanyaan dan model.
N_GENERATOR_WARMUP = 1
N_GENERATOR_REPEAT = 10  # Ulangan warm untuk SETIAP pertanyaan/model.
NUM_PREDICT = 384  # Generator terpisah; bukan pengaturan backend.
NUM_CTX = 8192
SEED = 42
OLLAMA_KEEP_ALIVE = "15m"

TEST_QUESTIONS = [
    "Berapa persentase penduduk lanjut usia di Indonesia menurut publikasi Statistik Penduduk Lanjut Usia edisi terbaru?",
    "Apa yang dimaksud dengan Indeks Khusus Penanganan Stunting (IKPS) menurut publikasi BPS?",
    "Bagaimana tren Indeks Pembangunan Manusia (IPM) Indonesia dari edisi ke edisi?",
    "Berapa Indeks Perilaku Anti Korupsi (IPAK) nasional pada edisi terakhir?",
]

# Satu CSV utama untuk seluruh pengukuran; JSON tetap dipakai untuk detail respons.
METRICS_CSV = OUT / "system_performance_all.csv"
METRIC_COLUMNS = [
    "run_id", "test_scope", "model_key", "ollama_tag", "scope", "use_rag", "phase",
    "question_id", "question_index", "question", "repeat", "status", "status_code",
    "error_type", "answer_status", "answer", "request_id", "response_style", "top_k",
    "expansion_s", "search_rerank_s", "retrieval_s", "generation_s", "citation_s",
    "first_token_s", "processing_s", "queue_s", "backend_total_s", "client_total_s",
    "client_overhead_s", "failure_wall_s", "ttft_approx_s", "load_s",
    "prompt_eval_s", "ollama_total_s", "eval_count", "prompt_eval_count",
    "tokens_per_s", "answer_tokens_per_s",
    "baseline_ram_mb", "peak_ram_mb", "ram_delta_mb",
    "baseline_gpu_mb", "peak_gpu_mb", "gpu_delta_mb",
    "baseline_system_ram_used_mb", "peak_system_ram_used_mb", "system_ram_delta_mb",
    "baseline_system_ram_available_mb", "min_system_ram_available_mb",
    "baseline_cpu_percent", "peak_cpu_percent", "ollama_process_count", "resource_samples",
    "ram_scope", "gpu_scope", "memory_unit", "sample_interval_s",
    "cache_enabled", "cache_expansion", "cache_search_rerank",
    "source_count", "answer_chars", "response_chars", "expanded_query_count",
    "expansion_note", "warnings", "citation_statuses", "kb_fingerprint", "num_ctx",
    "threshold", "ollama_load_duration", "ollama_prompt_eval_count",
    "ollama_prompt_eval_duration", "ollama_eval_count", "ollama_eval_duration",
    "ollama_total_duration", "ollama_done_reason",
]
METRIC_KEYS = ["test_scope", "model_key", "use_rag", "phase", "question_index", "repeat"]
ALL_METRIC_ROWS = []


def record_metric(row):
    record = {column: None for column in METRIC_COLUMNS}
    record.update(row)
    record.update(run_id=RUN_ID, ram_scope="Ollama process RSS; excludes backend RAM",
                  gpu_scope="Whole NVIDIA devices", memory_unit="MiB",
                  sample_interval_s=SAMPLE_INTERVAL_S)
    record["question_id"] = f"Q{int(record['question_index']):02d}"
    key = tuple(record.get(column) for column in METRIC_KEYS)
    # Mengulang sel benchmark pada sesi yang sama mengganti hasil kombinasi yang sama.
    index = next((i for i, previous in enumerate(ALL_METRIC_ROWS)
                  if tuple(previous.get(column) for column in METRIC_KEYS) == key), None)
    if index is None:
        ALL_METRIC_ROWS.append(record)
    else:
        ALL_METRIC_ROWS[index] = record
    frame = pd.DataFrame(ALL_METRIC_ROWS)
    columns = METRIC_COLUMNS + sorted(set(frame.columns) - set(METRIC_COLUMNS))
    temporary = METRICS_CSV.with_suffix(".csv.tmp")
    frame.reindex(columns=columns).to_csv(temporary, index=False, encoding="utf-8-sig")
    try:
        temporary.replace(METRICS_CSV)
    except PermissionError:
        raise PermissionError(
            f"Tutup file CSV di Excel sebelum melanjutkan: {METRICS_CSV}"
        ) from None


def metric_frame(test_scope=None):
    frame = pd.DataFrame(ALL_METRIC_ROWS, columns=METRIC_COLUMNS + sorted(
        {key for row in ALL_METRIC_ROWS for key in row} - set(METRIC_COLUMNS)))
    if test_scope is not None:
        frame = frame.loc[frame["test_scope"].eq(test_scope)]
    return frame.copy()


## 2. Preflight backend QA RAG dan Ollama

In [21]:
def normalize_ollama_name(name):
    return str(name).removesuffix(":latest")


def preflight():
    openapi_response = requests.get(f"{BASE_URL}/openapi.json", timeout=30)
    openapi_response.raise_for_status()
    paths = openapi_response.json().get("paths", {})
    required_paths = {"/api/health", "/api/ready", "/api/ask", "/api/ask/stream"}
    if E2E_MODE == "evaluation":
        required_paths.add("/api/eval/ask")
    missing_paths = required_paths - set(paths)
    if missing_paths:
        raise RuntimeError(f"Endpoint backend tidak lengkap: {sorted(missing_paths)}")

    health_response = requests.get(f"{BASE_URL}/api/health", timeout=30)
    health_response.raise_for_status()
    health = health_response.json()

    if not health.get("ready"):
        raise RuntimeError("Indeks backend belum siap.")
    ready_response = requests.get(f"{BASE_URL}/api/ready", timeout=30)
    ready_response.raise_for_status()

    tags_response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=30)
    tags_response.raise_for_status()
    tag_records = tags_response.json().get("models", [])
    available_tags = {
        str(record.get("name") or record.get("model"))
        for record in tag_records
        if record.get("name") or record.get("model")
    }
    normalized_tags = {normalize_ollama_name(tag): tag for tag in available_tags}

    missing_app_models = set(MODEL_KEY_TO_OLLAMA_TAG) - set(health.get("available_models", []))
    if missing_app_models:
        raise RuntimeError(f"Model belum terdaftar di backend: {sorted(missing_app_models)}")
    if not health.get("inference_reachable"):
        raise RuntimeError("Backend tidak dapat menjangkau Ollama.")

    resolved_tags = {}
    for model_key, configured_tag in MODEL_KEY_TO_OLLAMA_TAG.items():
        if configured_tag in available_tags:
            resolved_tags[model_key] = configured_tag
        elif normalize_ollama_name(configured_tag) in normalized_tags:
            resolved_tags[model_key] = normalized_tags[normalize_ollama_name(configured_tag)]
        else:
            raise RuntimeError(f"Tag Ollama untuk {model_key} tidak tersedia: {configured_tag}")

    snapshot = {
        "backend_url": BASE_URL,
        "ollama_url": OLLAMA_URL,
        "health": health,
        "scope": E2E_MODE,
        "api_version": openapi_response.json().get("info", {}).get("version"),
        "resolved_tags": resolved_tags,
        "available_tags": sorted(available_tags),
    }
    (OUT / "preflight.json").write_text(
        json.dumps(snapshot, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(json.dumps(snapshot, ensure_ascii=False, indent=2))
    return snapshot


PREFLIGHT = preflight()
RESOLVED_OLLAMA_TAGS = PREFLIGHT["resolved_tags"]


{
  "backend_url": "http://127.0.0.1:8000",
  "ollama_url": "http://localhost:11434",
  "health": {
    "ready": true,
    "inference_reachable": true,
    "available_models": [
      "llama3.2-base",
      "llama3.2-finetuned",
      "gemma2-base",
      "gemma2-finetuned"
    ],
    "default_model": "llama3.2-finetuned"
  },
  "scope": "production",
  "api_version": "0.2.0",
  "resolved_tags": {
    "llama3.2-finetuned": "llama3.2-finetuned:latest",
    "gemma2-finetuned": "gemma2-finetuned:latest"
  },
  "available_tags": [
    "gemma2-base:latest",
    "gemma2-finetuned:latest",
    "llama3.2-base:latest",
    "llama3.2-finetuned:latest"
  ]
}


## 3. Sampling CPU, GPU, dan RAM

Setiap request mencatat kondisi awal dan puncak. RAM proses Ollama dipisahkan dari RAM seluruh sistem, sehingga memori yang sudah dipakai aplikasi lain tidak keliru dianggap sebagai kebutuhan model. CPU dicatat sebagai utilisasi seluruh sistem karena beban awal komputer juga memengaruhi hasil waktu respons.

In [22]:
try:
    import pynvml

    pynvml.nvmlInit()
    GPU_HANDLES = [
        pynvml.nvmlDeviceGetHandleByIndex(index)
        for index in range(pynvml.nvmlDeviceGetCount())
    ]
    GPU_AVAILABLE = bool(GPU_HANDLES)
except Exception as exc:
    pynvml = None
    GPU_HANDLES = []
    GPU_AVAILABLE = False
    print(f"GPU NVIDIA tidak tersedia; kolom GPU akan kosong. Detail: {type(exc).__name__}: {exc}")


def find_ollama_processes():
    processes = []
    for process in psutil.process_iter(["pid", "name"]):
        try:
            if "ollama" in (process.info.get("name") or "").lower():
                processes.append(process)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            continue
    return processes


def current_resource_usage_mb():
    gpu_mb = np.nan
    if GPU_AVAILABLE:
        try:
            # NVML total device usage; hentikan beban GPU lain saat benchmark.
            gpu_mb = sum(
                pynvml.nvmlDeviceGetMemoryInfo(handle).used
                for handle in GPU_HANDLES
            ) / (1024 ** 2)
        except Exception:
            gpu_mb = np.nan

    ram_bytes = 0
    process_count = 0
    for process in find_ollama_processes():
        try:
            ram_bytes += process.memory_info().rss
            process_count += 1
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            continue
    ram_mb = ram_bytes / (1024 ** 2) if process_count else np.nan
    return gpu_mb, ram_mb, process_count


class ResourceSampler:
    def __init__(self, interval=SAMPLE_INTERVAL_S):
        self.interval = interval
        self.baseline_gpu_mb = np.nan
        self.baseline_ram_mb = np.nan
        self.peak_gpu_mb = np.nan
        self.peak_ram_mb = np.nan
        self.baseline_system_ram_used_mb = np.nan
        self.baseline_system_ram_available_mb = np.nan
        self.peak_system_ram_used_mb = np.nan
        self.min_system_ram_available_mb = np.nan
        self.baseline_cpu_percent = np.nan
        self.peak_cpu_percent = np.nan
        self.ollama_process_count = 0
        self.sample_count = 0
        self._stop = threading.Event()

    def _sample_once(self):
        gpu_mb, ram_mb, process_count = current_resource_usage_mb()
        memory = psutil.virtual_memory()
        system_ram_used_mb = memory.used / (1024 ** 2)
        system_ram_available_mb = memory.available / (1024 ** 2)
        cpu_percent = psutil.cpu_percent(interval=None)
        self.ollama_process_count = max(self.ollama_process_count, process_count)
        if not np.isnan(gpu_mb):
            self.peak_gpu_mb = gpu_mb if np.isnan(self.peak_gpu_mb) else max(self.peak_gpu_mb, gpu_mb)
        if not np.isnan(ram_mb):
            self.peak_ram_mb = ram_mb if np.isnan(self.peak_ram_mb) else max(self.peak_ram_mb, ram_mb)
        self.peak_system_ram_used_mb = (
            system_ram_used_mb if np.isnan(self.peak_system_ram_used_mb)
            else max(self.peak_system_ram_used_mb, system_ram_used_mb)
        )
        self.min_system_ram_available_mb = (
            system_ram_available_mb if np.isnan(self.min_system_ram_available_mb)
            else min(self.min_system_ram_available_mb, system_ram_available_mb)
        )
        self.peak_cpu_percent = (
            cpu_percent if np.isnan(self.peak_cpu_percent)
            else max(self.peak_cpu_percent, cpu_percent)
        )
        self.sample_count += 1

    def _run(self):
        while not self._stop.is_set():
            self._sample_once()
            self._stop.wait(self.interval)

    def __enter__(self):
        memory = psutil.virtual_memory()
        self.baseline_system_ram_used_mb = memory.used / (1024 ** 2)
        self.baseline_system_ram_available_mb = memory.available / (1024 ** 2)
        self.baseline_cpu_percent = psutil.cpu_percent(interval=0.2)
        self.baseline_gpu_mb, self.baseline_ram_mb, _ = current_resource_usage_mb()
        self.peak_gpu_mb = self.baseline_gpu_mb
        self.peak_ram_mb = self.baseline_ram_mb
        self.peak_system_ram_used_mb = self.baseline_system_ram_used_mb
        self.min_system_ram_available_mb = self.baseline_system_ram_available_mb
        self.peak_cpu_percent = self.baseline_cpu_percent
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self._stop.set()
        self._thread.join(timeout=max(1.0, self.interval * 4))
        self._sample_once()

    def as_dict(self):
        return {
            "baseline_gpu_mb": self.baseline_gpu_mb,
            "peak_gpu_mb": self.peak_gpu_mb,
            "gpu_delta_mb": self.peak_gpu_mb - self.baseline_gpu_mb
                if not np.isnan(self.peak_gpu_mb) and not np.isnan(self.baseline_gpu_mb) else np.nan,
            "baseline_ram_mb": self.baseline_ram_mb,
            "peak_ram_mb": self.peak_ram_mb,
            "ram_delta_mb": self.peak_ram_mb - self.baseline_ram_mb
                if not np.isnan(self.peak_ram_mb) and not np.isnan(self.baseline_ram_mb) else np.nan,
            "baseline_system_ram_used_mb": self.baseline_system_ram_used_mb,
            "baseline_system_ram_available_mb": self.baseline_system_ram_available_mb,
            "peak_system_ram_used_mb": self.peak_system_ram_used_mb,
            "system_ram_delta_mb": self.peak_system_ram_used_mb - self.baseline_system_ram_used_mb,
            "min_system_ram_available_mb": self.min_system_ram_available_mb,
            "baseline_cpu_percent": self.baseline_cpu_percent,
            "peak_cpu_percent": self.peak_cpu_percent,
            "ollama_process_count": self.ollama_process_count,
            "resource_samples": self.sample_count,
        }


def detect_cpu_name():
    candidates = [platform.processor(), platform.uname().processor]
    if platform.system() == "Windows":
        try:
            import winreg
            with winreg.OpenKey(
                winreg.HKEY_LOCAL_MACHINE,
                "HARDWARE\\DESCRIPTION\\System\\CentralProcessor\\0",
            ) as key:
                candidates.insert(0, winreg.QueryValueEx(key, "ProcessorNameString")[0])
        except Exception:
            pass
        try:
            output = subprocess.check_output(
                [
                    "powershell", "-NoProfile", "-Command",
                    "(Get-CimInstance Win32_Processor | Select-Object -First 1 -ExpandProperty Name)",
                ],
                text=True,
                timeout=15,
                stderr=subprocess.DEVNULL,
            ).strip()
            candidates.insert(0, output)
        except Exception:
            pass
    elif Path("/proc/cpuinfo").is_file():
        try:
            for line in Path("/proc/cpuinfo").read_text(encoding="utf-8", errors="ignore").splitlines():
                if line.lower().startswith("model name"):
                    candidates.insert(0, line.split(":", 1)[1].strip())
                    break
        except Exception:
            pass
    return next((str(name).strip() for name in candidates if str(name).strip()), "unknown")


def detect_gpus():
    devices = []
    if GPU_AVAILABLE:
        driver = pynvml.nvmlSystemGetDriverVersion()
        if isinstance(driver, bytes):
            driver = driver.decode(errors="replace")
        for index, handle in enumerate(GPU_HANDLES):
            name = pynvml.nvmlDeviceGetName(handle)
            if isinstance(name, bytes):
                name = name.decode(errors="replace")
            memory = pynvml.nvmlDeviceGetMemoryInfo(handle)
            utilization = pynvml.nvmlDeviceGetUtilizationRates(handle)
            devices.append({
                "index": index,
                "name": str(name),
                "vram_total_gb": round(memory.total / (1024 ** 3), 2),
                "vram_used_at_inventory_mb": round(memory.used / (1024 ** 2), 1),
                "utilization_at_inventory_percent": float(utilization.gpu),
                "driver_version": str(driver),
                "measurement_available": True,
            })
    else:
        # Inventaris NVIDIA masih dapat dibaca lewat CLI walau binding pynvml
        # tidak terpasang. Ini hanya fallback identifikasi, bukan sampling 50 ms.
        try:
            output = subprocess.check_output(
                [
                    "nvidia-smi",
                    "--query-gpu=index,name,memory.total,memory.used,utilization.gpu,driver_version",
                    "--format=csv,noheader,nounits",
                ],
                text=True,
                timeout=15,
                stderr=subprocess.DEVNULL,
            ).strip()
            for line in output.splitlines():
                index, name, memory_mib, used_mib, utilization, driver = [
                    part.strip() for part in line.split(",", 5)
                ]
                devices.append({
                    "index": int(index),
                    "name": name,
                    "vram_total_gb": round(float(memory_mib) / 1024, 2),
                    "vram_used_at_inventory_mb": round(float(used_mib), 1),
                    "utilization_at_inventory_percent": float(utilization),
                    "driver_version": driver,
                    "measurement_available": False,
                    "note": "Terdeteksi via nvidia-smi; instal nvidia-ml-py untuk sampling 50 ms.",
                })
        except Exception:
            pass

    if not devices and platform.system() == "Windows":
        try:
            command = (
                "Get-CimInstance Win32_VideoController | "
                "Select-Object Name,AdapterRAM,DriverVersion | ConvertTo-Json -Compress"
            )
            output = subprocess.check_output(
                ["powershell", "-NoProfile", "-Command", command],
                text=True,
                timeout=15,
                stderr=subprocess.DEVNULL,
            ).strip()
            parsed = json.loads(output) if output else []
            if isinstance(parsed, dict):
                parsed = [parsed]
            for index, item in enumerate(parsed):
                adapter_ram = item.get("AdapterRAM")
                devices.append({
                    "index": index,
                    "name": item.get("Name"),
                    "vram_total_gb": round(adapter_ram / (1024 ** 3), 2) if adapter_ram else None,
                    "driver_version": item.get("DriverVersion"),
                    "measurement_available": False,
                    "note": "Fallback Windows; AdapterRAM dapat tidak akurat untuk VRAM di atas 4 GB.",
                })
        except Exception:
            pass
    return devices


def detect_hardware():
    memory = psutil.virtual_memory()
    cpu_frequency = psutil.cpu_freq()
    cpu_used_percent = psutil.cpu_percent(interval=0.2)
    inventory = {
        "cpu": {
            "name": detect_cpu_name(),
            "physical_cores": psutil.cpu_count(logical=False),
            "logical_cores": psutil.cpu_count(logical=True),
            "max_frequency_mhz": round(cpu_frequency.max, 1) if cpu_frequency else None,
            "utilization_at_inventory_percent": cpu_used_percent,
        },
        "ram": {
            "total_bytes": int(memory.total),
            "total_gb": round(memory.total / (1024 ** 3), 2),
            "used_at_inventory_gb": round(memory.used / (1024 ** 3), 2),
            "available_at_inventory_gb": round(memory.available / (1024 ** 3), 2),
            "used_at_inventory_percent": float(memory.percent),
        },
        "gpus": detect_gpus(),
        "operating_system": platform.platform(),
        "python_version": platform.python_version(),
        "hardware_notes": HARDWARE_NOTES,
    }
    (OUT / "hardware_inventory.json").write_text(
        json.dumps(inventory, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return inventory


HARDWARE_INVENTORY = detect_hardware()
print(json.dumps(HARDWARE_INVENTORY, ensure_ascii=False, indent=2))


GPU NVIDIA tidak tersedia; kolom GPU akan kosong. Detail: ModuleNotFoundError: No module named 'pynvml'
{
  "cpu": {
    "name": "11th Gen Intel(R) Core(TM) i5-11300H @ 3.10GHz",
    "physical_cores": 4,
    "logical_cores": 8,
    "max_frequency_mhz": 3110.0,
    "utilization_at_inventory_percent": 18.9
  },
  "ram": {
    "total_bytes": 16952647680,
    "total_gb": 15.79,
    "used_at_inventory_gb": 15.14,
    "available_at_inventory_gb": 0.65,
    "used_at_inventory_percent": 95.9
  },
  "gpus": [
    {
      "index": 0,
      "name": "NVIDIA GeForce RTX 3050 Laptop GPU",
      "vram_total_gb": 4.0,
      "vram_used_at_inventory_mb": 2365.0,
      "utilization_at_inventory_percent": 0.0,
      "driver_version": "566.14",
      "measurement_available": false,
      "note": "Terdeteksi via nvidia-smi; instal nvidia-ml-py untuk sampling 50 ms."
    }
  ],
  "operating_system": "Windows-10-10.0.26200-SP0",
  "python_version": "3.11.16",
  "hardware_notes": ""
}


## 4. Probe kontrak aktual `/api/ask`

In [23]:
def ask_api(question, model_key, use_rag=True, top_k=TOP_K, response_style=RESPONSE_STYLE):
    if E2E_MODE == "production" and not use_rag:
        raise ValueError("Non-RAG hanya melalui E2E_MODE='evaluation'.")
    headers = {}
    payload = dict(question=question, model_key=model_key, use_rag=use_rag,
                   top_k=top_k, response_style=response_style)
    endpoint = "/api/ask"
    if E2E_MODE == "evaluation":
        key = os.environ.get("EVALUATION_API_KEY", "")
        if len(key) < 32:
            raise ValueError("Set EVALUATION_API_KEY di environment kernel, minimal 32 karakter.")
        headers["X-Evaluation-Key"] = key
        payload["strict_expansion"] = True  # Tetap sama dengan perilaku UI.
        endpoint = "/api/eval/ask"
    started = time.perf_counter()
    response = requests.post(BASE_URL + endpoint, json=payload, headers=headers, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    body = response.json()
    return body, time.perf_counter() - started


def validate_ask_response(body, model_key, use_rag, evaluation=None):
    if evaluation is None:
        evaluation = E2E_MODE == "evaluation"
    required = {"answer", "answer_status", "sources", "other_sources", "latency",
                "config", "process", "cache", "generation_metrics", "warnings", "request_id"}
    if missing := required - set(body):
        raise ValueError(f"Respons kehilangan field: {sorted(missing)}")
    if not isinstance(body["answer"], str) or not body["answer"].strip():
        raise ValueError("answer kosong atau bukan string.")
    if not isinstance(body["sources"], list):
        raise ValueError("sources bukan list.")
    required_latency = {"expansion_s", "search_rerank_s", "retrieval_s", "citation_s",
                        "generation_s", "first_token_s", "processing_s", "queue_s", "total_s"}
    if missing := required_latency - set(body["latency"]):
        raise ValueError(f"latency kehilangan field: {sorted(missing)}")
    cfg = body["config"]
    if cfg.get("model_key") != model_key or cfg.get("use_rag") is not use_rag:
        raise ValueError("Konfigurasi respons berbeda dari permintaan.")
    if cfg.get("evaluation") is not evaluation:
        raise ValueError("Lingkup respons berbeda dari endpoint yang diminta.")
    if not use_rag and body["sources"]:
        raise ValueError("Non-RAG tidak boleh mengembalikan sumber retrieval.")
    if evaluation:
        if body["cache"].get("enabled") is not False:
            raise ValueError("Cache aplikasi masih aktif pada evaluasi.")
        if not isinstance(body.get("contexts"), list):
            raise ValueError("Evaluasi harus mengembalikan contexts asli.")


def save_response(name, body, **metadata):
    # Tidak menyimpan header, environment lengkap, atau API key.
    artifact = dict(metadata, response=body)
    (OUT / name).write_text(json.dumps(artifact, ensure_ascii=False, indent=2), encoding="utf-8")


PROBE_QUESTION = TEST_QUESTIONS[1]
if RUN_API_PROBE:
    model_key = next(iter(MODEL_KEY_TO_OLLAMA_TAG))
    probe_body, probe_client_s = ask_api(PROBE_QUESTION, model_key)
    validate_ask_response(probe_body, model_key, True)
    save_response("api_ask_probe.json", probe_body, question=PROBE_QUESTION,
                  client_total_s=probe_client_s, scope=E2E_MODE)
    print("Probe tersimpan:", OUT / "api_ask_probe.json")
    display(pd.DataFrame(probe_body["sources"]))


Probe tersimpan: D:\Politeknik Statistika STIS\Pembelajaran\Skripsi\SourceCode\SkripsiNabil\Hasil\tujuan3\20260927-214705-5142d6fe\api_ask_probe.json


,source_id,chunk_id,document_title,document_year,section_id,page_start,page_end,bps_url,score,index,answer,quote,citation_status
0,faiss:3729,laporan-indeks-khusus-penanganan-stunting--ikp...,Laporan Indeks Khusus Penanganan Stunting (IKP...,2024,sec_008,24,24,https://www.bps.go.id/id/publication/2024/12/1...,4.266599,1,,Indeks Khusus Penanganan Stunting (IKPS) disus...,excerpt_only
1,faiss:3731,laporan-indeks-khusus-penanganan-stunting--ikp...,Laporan Indeks Khusus Penanganan Stunting (IKP...,2024,sec_009,25,28,https://www.bps.go.id/id/publication/2024/12/1...,4.044086,2,,Laporan Indeks Khusus Penanganan Stunting 2022...,excerpt_only
2,faiss:3841,laporan-indeks-khusus-penanganan-stunting-2021...,Laporan Indeks Khusus Penanganan Stunting 2021...,2023,sec_008,24,25,https://www.bps.go.id/id/publication/2023/12/1...,3.545054,3,,Hal ini dikarenakan IKPS merupakan bagian dari...,excerpt_only
3,faiss:3983,laporan-indeks-khusus-penanganan-stunting-kabu...,Laporan Indeks Khusus Penanganan Stunting Kabu...,2023,sec_034,52,53,https://www.bps.go.id/id/publication/2023/12/1...,3.481221,4,,Hal ini dikarenakan IKPS merupakan bagian dari...,excerpt_only
4,faiss:3840,laporan-indeks-khusus-penanganan-stunting-2021...,Laporan Indeks Khusus Penanganan Stunting 2021...,2023,sec_007,23,24,https://www.bps.go.id/id/publication/2023/12/1...,3.218345,5,,Laporan Indeks Khusus Penanganan Stunting 2021...,excerpt_only


## 5. Uji fungsi endpoint

In [24]:
def run_functional_tests():
    rows = []
    model_key = next(iter(MODEL_KEY_TO_OLLAMA_TAG))
    invalid_cases = [
        ("empty_question", {"question": ""}, {422}),
        ("whitespace_question", {"question": "   "}, {422}),
        ("unknown_model", {"question": "Apa itu BPS?", "model_key": "model-tidak-ada"}, {400}),
        ("public_non_rag_rejected", {"question": "Apa itu BPS?", "use_rag": False}, {422}),
        ("invalid_top_k", {"question": "Apa itu BPS?", "top_k": 0}, {422}),
    ]
    for case, payload, expected in invalid_cases:
        try:
            response = requests.post(BASE_URL + "/api/ask", json=payload, timeout=30)
            rows.append(dict(case=case, status="pass" if response.status_code in expected else "fail",
                             status_code=response.status_code, detail=response.text[:1000]))
        except requests.RequestException as exc:
            rows.append(dict(case=case, status="error", detail=type(exc).__name__))
    for model_key in MODEL_KEY_TO_OLLAMA_TAG:
        for use_rag in E2E_RAG_MODES:
            case = f"valid_schema__{E2E_MODE}__{model_key}__rag_{use_rag}"
            try:
                body, elapsed = ask_api(PROBE_QUESTION, model_key, use_rag=use_rag)
                validate_ask_response(body, model_key, use_rag)
                rows.append(dict(case=case, status="pass", status_code=200, client_total_s=elapsed,
                                 answer_status=body["answer_status"], detail="Kontrak valid; bukan penilaian akurasi."))
            except Exception as exc:
                rows.append(dict(case=case, status="error", detail=type(exc).__name__))
    frame = pd.DataFrame(rows)
    (OUT / "functional_results.json").write_text(
        frame.to_json(orient="records", force_ascii=False, indent=2), encoding="utf-8")
    return frame


def iter_sse(response):
    event, data = "message", []
    # Chunk kecil mengurangi buffering klien saat mengukur token pertama.
    for line in response.iter_lines(chunk_size=1, decode_unicode=True):
        if isinstance(line, bytes):
            line = line.decode("utf-8")
        if not line:
            if data:
                yield event, json.loads("\n".join(data))
            event, data = "message", []
        elif line.startswith("event:"):
            event = line[6:].strip()
        elif line.startswith("data:"):
            data.append(line[5:].lstrip(" "))
    if data:
        raise RuntimeError("SSE terputus di tengah event.")


def run_stream_stage_test():
    model_key = next(iter(MODEL_KEY_TO_OLLAMA_TAG))
    payload = dict(question=PROBE_QUESTION, model_key=model_key, use_rag=True,
                   top_k=TOP_K, response_style=RESPONSE_STYLE)
    started = time.perf_counter()
    events, first_token_s, answer_s, final, error = [], None, None, None, None
    with requests.post(BASE_URL + "/api/ask/stream", json=payload, stream=True,
                       timeout=REQUEST_TIMEOUT) as response:
        response.raise_for_status()
        response.encoding = "utf-8"
        for event, body in iter_sse(response):
            elapsed = time.perf_counter() - started
            events.append(dict(event=event, elapsed_s=elapsed))
            if event == "token" and body.get("text") and first_token_s is None:
                first_token_s = elapsed
            if event == "answer":
                answer_s = elapsed
            if event == "error":
                error = body
                break
            if event == "done":
                final = body
                break
    elapsed = time.perf_counter() - started
    names = [item["event"] for item in events]
    issues = []
    if error or final is None:
        issues.append("Stream error atau terputus tanpa done.")
    if final is not None:
        validate_ask_response(final, model_key, True, evaluation=False)
        for required in ("expanded", "reranked", "answer", "done"):
            if required not in names:
                issues.append(f"Event {required} tidak ditemukan.")
        if "citing" in names and ("answer" not in names or names.index("answer") > names.index("citing")):
            issues.append("Ringkasan sumber dimulai sebelum jawaban.")
        if final["answer_status"] != "no_evidence" and first_token_s is None:
            issues.append("Token draf tidak diterima.")
    result = dict(status="fail" if issues else "pass", issues=issues, events=events,
                  client_first_draft_token_s=first_token_s, client_checked_answer_s=answer_s,
                  client_total_s=elapsed, response=final, error=error, scope="production_stream")
    (OUT / "stream_stage_test.json").write_text(
        json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    return {k: v for k, v in result.items() if k != "response"}


if RUN_FUNCTIONAL_TESTS:
    display(run_functional_tests())
if RUN_STREAM_TEST:
    print(json.dumps(run_stream_stage_test(), ensure_ascii=False, indent=2))
if not RUN_FUNCTIONAL_TESTS and not RUN_STREAM_TEST:
    print("Uji fungsi dan streaming belum diaktifkan.")


,case,status,status_code,detail,client_total_s,answer_status
0,empty_question,pass,422,"{""detail"":[{""type"":""string_too_short"",""loc"":[""...",NaN,NaN
1,whitespace_question,pass,422,"{""detail"":[{""type"":""string_too_short"",""loc"":[""...",NaN,NaN
2,unknown_model,pass,400,"{""detail"":""Model tidak dikenal.""}",NaN,NaN
3,public_non_rag_rejected,pass,422,"{""detail"":[{""type"":""literal_error"",""loc"":[""bod...",NaN,NaN
4,invalid_top_k,pass,422,"{""detail"":[{""type"":""greater_than_equal"",""loc"":...",NaN,NaN
5,valid_schema__production__llama3.2-finetuned__...,pass,200,Kontrak valid; bukan penilaian akurasi.,74.777007,unverified
6,valid_schema__production__gemma2-finetuned__ra...,pass,200,Kontrak valid; bukan penilaian akurasi.,29.787763,unverified


{
  "status": "pass",
  "issues": [],
  "events": [
    {
      "event": "queued",
      "elapsed_s": 0.015182999995886348
    },
    {
      "event": "expanding",
      "elapsed_s": 0.017145000005257316
    },
    {
      "event": "expanded",
      "elapsed_s": 0.021244100003968924
    },
    {
      "event": "retrieving",
      "elapsed_s": 0.021727400002419017
    },
    {
      "event": "reranked",
      "elapsed_s": 0.1231746999983443
    },
    {
      "event": "sources",
      "elapsed_s": 0.1404121999948984
    },
    {
      "event": "generating",
      "elapsed_s": 0.1406416999961948
    },
    {
      "event": "heartbeat",
      "elapsed_s": 5.069665999995777
    },
    {
      "event": "token",
      "elapsed_s": 9.049663199999486
    },
    {
      "event": "token",
      "elapsed_s": 9.08658750000177
    },
    {
      "event": "token",
      "elapsed_s": 9.129775199995493
    },
    {
      "event": "token",
      "elapsed_s": 9.164926900004502
    },
    {
      "event"

## 6. Benchmark end-to-end QA RAG melalui backend

In [25]:
def benchmark_e2e_model(model_key, use_rag=True):
    # Warmup bukan baris pengukuran; kegagalan warmup tidak menghilangkan pertanyaan uji.
    for warmup_index in range(N_E2E_WARMUP):
        try:
            body, _ = ask_api(TEST_QUESTIONS[warmup_index % len(TEST_QUESTIONS)], model_key, use_rag=use_rag)
            validate_ask_response(body, model_key, use_rag)
        except Exception as exc:
            print(f"Warmup {model_key} gagal: {type(exc).__name__}; pengukuran tetap dilanjutkan.")
    rows = []
    for repeat in range(1, N_E2E_REPEAT + 1):
        for question_index, question in enumerate(TEST_QUESTIONS, start=1):
            row = dict(test_scope="e2e_system", model_key=model_key, scope=E2E_MODE,
                       use_rag=use_rag, phase="measured", repeat=repeat,
                       question_index=question_index, question=question,
                       response_style=RESPONSE_STYLE, top_k=TOP_K,
                       answer_status="request_failed", status="error")
            sampler = ResourceSampler()
            request_started = time.perf_counter()
            try:
                with sampler:
                    request_started = time.perf_counter()
                    body, client_total_s = ask_api(question, model_key, use_rag=use_rag)
                # Simpan respons sebelum validasi agar kesalahan kontrak bisa ditelusuri.
                save_response(f"response_{model_key}_rag{use_rag}_r{repeat}_q{question_index}.json",
                              body, question=question, client_total_s=client_total_s, scope=E2E_MODE)
                validate_ask_response(body, model_key, use_rag)
                latency, process, cache = body["latency"], body["process"], body["cache"]
                metrics = body["generation_metrics"]
                eval_s = metrics.get("eval_duration", 0) / 1e9
                row.update(latency)
                row.update(client_total_s=client_total_s, backend_total_s=latency["total_s"],
                           client_overhead_s=client_total_s-latency["total_s"],
                           source_count=len(body["sources"]), answer_chars=len(body["answer"]),
                           answer=body["answer"], answer_status=body["answer_status"],
                           request_id=body["request_id"],
                           cache_expansion=cache["expansion"], cache_search_rerank=cache["search_rerank"],
                           cache_enabled=cache["enabled"], kb_fingerprint=body["config"]["kb_fingerprint"],
                           num_ctx=body["config"]["num_ctx"], threshold=body["config"]["threshold"],
                           expanded_query_count=len(process.get("expanded_queries", [])),
                           expansion_note=process.get("expansion_note"),
                           warnings=json.dumps(body["warnings"], ensure_ascii=False),
                           citation_statuses=json.dumps([s.get("citation_status") for s in body["sources"]]),
                           answer_tokens_per_s=metrics.get("eval_count", 0)/eval_s if eval_s else np.nan,
                           status="ok", status_code=200)
                row.update({f"ollama_{k}": v for k, v in metrics.items()})
            except Exception as exc:
                row.update(status="error", error_type=type(exc).__name__,
                           failure_wall_s=time.perf_counter()-request_started)
                response = getattr(exc, "response", None)
                if response is not None:
                    row["status_code"] = response.status_code
            row.update(sampler.as_dict())
            record_metric(row)  # Simpan setiap request, termasuk yang gagal.
            rows.append(row)
            print(f"{model_key} RAG={use_rag}, r{repeat} q{question_index}/{len(TEST_QUESTIONS)}: {row['status']}")
    return rows


def percentile_95(series):
    return series.quantile(0.95)


def summarize_e2e(frame):
    # Ringkasan ditampilkan saja, tidak membuat CSV metrik lain.
    if frame.empty:
        return frame
    keys = ["model_key", "scope", "use_rag", "status", "answer_status",
            "cache_expansion", "cache_search_rerank"]
    return frame.groupby(keys, dropna=False).agg(
        n=("repeat", "count"), question_count=("question_index", "nunique"),
        client_median_s=("client_total_s", "median"),
        client_p95_s=("client_total_s", percentile_95),
        backend_median_s=("backend_total_s", "median"),
        retrieval_median_s=("retrieval_s", "median"),
        retrieval_p95_s=("retrieval_s", percentile_95),
        citation_median_s=("citation_s", "median"),
        generation_median_s=("generation_s", "median"),
        generation_p95_s=("generation_s", percentile_95),
        peak_gpu_mb=("peak_gpu_mb", "max"),
        peak_ram_mb=("peak_ram_mb", "max"),
        peak_system_ram_used_mb=("peak_system_ram_used_mb", "max"),
        peak_cpu_percent=("peak_cpu_percent", "max"),
    ).reset_index()


def check_metric_coverage():
    expected = set()
    if RUN_E2E_BENCHMARK:
        for model_key in MODEL_KEY_TO_OLLAMA_TAG:
            for use_rag in E2E_RAG_MODES:
                for repeat in range(1, N_E2E_REPEAT + 1):
                    for qi in range(1, len(TEST_QUESTIONS) + 1):
                        expected.add(("e2e_system", model_key, use_rag, "measured", qi, repeat))
    if RUN_GENERATOR_BENCHMARK:
        for model_key in MODEL_KEY_TO_OLLAMA_TAG:
            for qi in range(1, len(TEST_QUESTIONS) + 1):
                expected.add(("generator_only", model_key, False, "model_unloaded", qi, 1))
                for repeat in range(1, N_GENERATOR_REPEAT + 1):
                    expected.add(("generator_only", model_key, False, "warm", qi, repeat))
    actual = {tuple(row.get(k) for k in METRIC_KEYS) for row in ALL_METRIC_ROWS}
    missing = expected - actual
    report = dict(expected_rows=len(expected), recorded_rows=len(actual & expected),
                  missing_rows=[list(key) for key in sorted(missing)],
                  error_rows=sum(row.get("status") == "error" for row in ALL_METRIC_ROWS))
    (OUT / "coverage.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    if missing:
        raise RuntimeError(f"{len(missing)} pengukuran belum tercatat; lihat coverage.json.")
    print(f"Cakupan lengkap: {report['recorded_rows']}/{report['expected_rows']} pengukuran; "
          f"{report['error_rows']} gagal. Gagal tetap tercatat, bukan dianggap berhasil.")
    return report


## 7. Benchmark generator langsung melalui Ollama

In [26]:
def ollama_generate(ollama_tag, prompt, keep_alive=OLLAMA_KEEP_ALIVE):
    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": ollama_tag,
            "prompt": prompt,
            "stream": False,
            "keep_alive": keep_alive,
            "options": {"temperature": 0, "num_predict": NUM_PREDICT, "num_ctx": NUM_CTX, "seed": SEED},
        },
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()
    body = response.json()
    required = {
        "load_duration", "prompt_eval_count", "prompt_eval_duration",
        "eval_count", "eval_duration", "total_duration",
    }
    if missing := required - set(body):
        raise ValueError(f"Respons Ollama kehilangan metrik: {sorted(missing)}")
    return body


def unload_ollama_model(ollama_tag):
    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={"model": ollama_tag, "keep_alive": 0},
        timeout=60,
    )
    response.raise_for_status()
    time.sleep(1)


def measure_generator_once(model_key, ollama_tag, prompt, phase, repeat, prompt_index):
    with ResourceSampler() as sampler:
        started = time.perf_counter()
        body = ollama_generate(ollama_tag, prompt)
        client_total_s = time.perf_counter() - started

    load_s = body["load_duration"] / 1e9
    prompt_eval_s = body["prompt_eval_duration"] / 1e9
    generation_s = body["eval_duration"] / 1e9
    eval_count = int(body["eval_count"])
    return {
        "model_key": model_key,
        "ollama_tag": ollama_tag,
        "phase": phase,
        "repeat": repeat,
        "prompt_index": prompt_index,
        "prompt_chars": len(prompt),
        "prompt_eval_count": int(body["prompt_eval_count"]),
        "eval_count": eval_count,
        "ttft_approx_s": load_s + prompt_eval_s,
        "load_s": load_s,
        "prompt_eval_s": prompt_eval_s,
        "generation_s": generation_s,
        "tokens_per_s": eval_count / generation_s if generation_s > 0 else np.nan,
        "ollama_total_s": body["total_duration"] / 1e9,
        "client_total_s": client_total_s,
        "response_chars": len(str(body.get("response", ""))),
        "answer": str(body.get("response", "")),
        "status": "ok",
        **sampler.as_dict(),
    }


def benchmark_generator_model(model_key, ollama_tag):
    rows = []
    # Setiap pertanyaan memperoleh jumlah ulangan yang sama.
    for question_index, question in enumerate(TEST_QUESTIONS, start=1):
        try:
            unload_ollama_model(ollama_tag)
            cold_error = None
        except Exception as exc:
            cold_error = type(exc).__name__
        schedules = [("model_unloaded", 1)] + [("warm", i) for i in range(1, N_GENERATOR_REPEAT + 1)]
        for phase, repeat in schedules:
            if phase == "warm" and repeat == 1:
                for _ in range(N_GENERATOR_WARMUP):
                    try:
                        ollama_generate(ollama_tag, question)
                    except Exception as exc:
                        print(f"Warmup generator gagal: {type(exc).__name__}")
            row = dict(test_scope="generator_only", scope="ollama_generate", use_rag=False,
                       model_key=model_key, ollama_tag=ollama_tag, phase=phase, repeat=repeat,
                       question_index=question_index, question=question,
                       num_ctx=NUM_CTX, num_predict=NUM_PREDICT, seed=SEED, status="error")
            started = time.perf_counter()
            if phase == "model_unloaded" and cold_error:
                row.update(error_type="UnloadFailed:" + cold_error)
            else:
                try:
                    row.update(measure_generator_once(model_key, ollama_tag, question,
                                                      phase, repeat, question_index))
                    row["status_code"] = 200
                except Exception as exc:
                    row.update(error_type=type(exc).__name__, failure_wall_s=time.perf_counter()-started)
                    response = getattr(exc, "response", None)
                    if response is not None:
                        row["status_code"] = response.status_code
            record_metric(row)
            rows.append(row)
            print(f"{model_key} generator q{question_index}/{len(TEST_QUESTIONS)} {phase} r{repeat}: {row['status']}")
    return rows


def summarize_generator(frame):
    return frame.groupby(["model_key", "phase"]).agg(
        n=("repeat", "count"),
        ttft_approx_median_s=("ttft_approx_s", "median"),
        ttft_approx_p95_s=("ttft_approx_s", percentile_95),
        tokens_per_s_median=("tokens_per_s", "median"),
        tokens_per_s_p95=("tokens_per_s", percentile_95),
        ollama_total_median_s=("ollama_total_s", "median"),
        client_total_median_s=("client_total_s", "median"),
        peak_gpu_mb=("peak_gpu_mb", "max"),
        peak_ram_mb=("peak_ram_mb", "max"),
        peak_system_ram_used_mb=("peak_system_ram_used_mb", "max"),
        max_system_ram_delta_mb=("system_ram_delta_mb", "max"),
        peak_cpu_percent=("peak_cpu_percent", "max"),
    ).reset_index()


## 8. Jalankan benchmark dan simpan seluruh hasil

In [27]:
environment = {
    "hardware_inventory": HARDWARE_INVENTORY,
    "run_id": RUN_ID,
    "e2e_mode": E2E_MODE,
    "e2e_rag_modes": E2E_RAG_MODES,
    "test_questions": TEST_QUESTIONS,
    "num_ctx_generator": NUM_CTX,
    "seed_generator": SEED,
    "keep_alive_generator": OLLAMA_KEEP_ALIVE,
    "api_version": PREFLIGHT["api_version"],
    "local_source_sha256": {
        str(p.relative_to(PROJECT_ROOT)): hashlib.sha256(p.read_bytes()).hexdigest()
        for p in sorted((PROJECT_ROOT / "app").glob("*.py"))
    },
    "source_hash_note": "Kode lokal notebook; pastikan sama dengan backend bila memakai server lain.",
    "package_versions": {name: __import__("importlib.metadata", fromlist=["version"]).version(name)
                         for name in ("requests", "numpy", "pandas", "psutil")},
    "backend_url": BASE_URL,
    "ollama_url": OLLAMA_URL,
    "model_tags": RESOLVED_OLLAMA_TAGS,
    "gpu_available": GPU_AVAILABLE,
    "gpu_measurement": "Total used memory across NVIDIA devices; keep other GPU workloads closed.",
    "ram_measurement": "Ollama RSS plus baseline/peak/available RAM for the entire system.",
    "cpu_measurement": "Whole-system utilization at baseline and peak during each request.",
    "sample_interval_s": SAMPLE_INTERVAL_S,
    "request_timeout": REQUEST_TIMEOUT,
    "top_k": TOP_K,
    "response_style": RESPONSE_STYLE,
    "e2e_warmup": N_E2E_WARMUP,
    "e2e_repeat_per_question": N_E2E_REPEAT,
    "generator_warmup": N_GENERATOR_WARMUP,
     "generator_repeat_per_question": N_GENERATOR_REPEAT,
    "main_metrics_csv": METRICS_CSV.name,
    "question_count": len(TEST_QUESTIONS),
    "num_predict": NUM_PREDICT,
    "ttft_definition": "load_duration + prompt_eval_duration; approximation from non-streaming Ollama metadata",
    "backend_latency_definition": {
        "retrieval_s": "query expansion + dense/BM25 + RRF + reranking",
        "citation_s": "citation generation",
        "generation_s": "final answer generation",
        "first_token_s": "server time to first draft answer token, includes queue",
        "processing_s": "backend pipeline excluding queue",
        "queue_s": "backend queue waiting time",
        "total_s": "backend pipeline including queue",
    },
    "deepseek_excluded": "Commercial API; local GPU/RAM and Ollama token/s are not comparable.",
}
(OUT / "system_performance_environment.json").write_text(
    json.dumps(environment, ensure_ascii=False, indent=2), encoding="utf-8"
)

if RUN_E2E_BENCHMARK:
    for model_key in MODEL_KEY_TO_OLLAMA_TAG:
        for use_rag in E2E_RAG_MODES:
            benchmark_e2e_model(model_key, use_rag=use_rag)
    e2e_raw = metric_frame("e2e_system")
    display(e2e_raw[["question_id", "question", "model_key", "repeat", "status",
                    "retrieval_s", "generation_s", "citation_s", "client_total_s",
                    "peak_ram_mb", "peak_gpu_mb"]])
    display(summarize_e2e(e2e_raw))
else:
    print("Benchmark sistem belum dijalankan.")

if RUN_GENERATOR_BENCHMARK:
    for model_key, ollama_tag in RESOLVED_OLLAMA_TAGS.items():
        benchmark_generator_model(model_key, ollama_tag)
    generator_raw = metric_frame("generator_only")
    successful_generator = generator_raw.loc[generator_raw["status"].eq("ok")]
    if not successful_generator.empty:
        display(summarize_generator(successful_generator))
else:
    print("Benchmark generator terpisah belum dijalankan.")

coverage = check_metric_coverage()
if ALL_METRIC_ROWS:
    print("CSV utama:", METRICS_CSV.resolve())
    print("Jumlah baris:", len(ALL_METRIC_ROWS), "| Pertanyaan:", len(TEST_QUESTIONS))
else:
    print("Belum ada hasil metrik. Aktifkan RUN_E2E_BENCHMARK.")


llama3.2-finetuned RAG=True, r1 q1/4: ok
llama3.2-finetuned RAG=True, r1 q2/4: ok
llama3.2-finetuned RAG=True, r1 q3/4: ok
llama3.2-finetuned RAG=True, r1 q4/4: ok
llama3.2-finetuned RAG=True, r2 q1/4: ok
llama3.2-finetuned RAG=True, r2 q2/4: ok
llama3.2-finetuned RAG=True, r2 q3/4: ok
llama3.2-finetuned RAG=True, r2 q4/4: ok
llama3.2-finetuned RAG=True, r3 q1/4: ok
llama3.2-finetuned RAG=True, r3 q2/4: ok
llama3.2-finetuned RAG=True, r3 q3/4: ok
llama3.2-finetuned RAG=True, r3 q4/4: ok
gemma2-finetuned RAG=True, r1 q1/4: ok
gemma2-finetuned RAG=True, r1 q2/4: ok
gemma2-finetuned RAG=True, r1 q3/4: ok
gemma2-finetuned RAG=True, r1 q4/4: ok
gemma2-finetuned RAG=True, r2 q1/4: ok
gemma2-finetuned RAG=True, r2 q2/4: ok
gemma2-finetuned RAG=True, r2 q3/4: ok
gemma2-finetuned RAG=True, r2 q4/4: ok
gemma2-finetuned RAG=True, r3 q1/4: ok
gemma2-finetuned RAG=True, r3 q2/4: ok
gemma2-finetuned RAG=True, r3 q3/4: ok
gemma2-finetuned RAG=True, r3 q4/4: ok


,question_id,question,model_key,repeat,status,retrieval_s,generation_s,citation_s,client_total_s,peak_ram_mb,peak_gpu_mb
0,Q01,Berapa persentase penduduk lanjut usia di Indo...,llama3.2-finetuned,1,ok,0.001,4.672,39.160,43.864569,38.988281,NaN
1,Q02,Apa yang dimaksud dengan Indeks Khusus Penanga...,llama3.2-finetuned,1,ok,0.001,19.342,37.712,57.065662,38.550781,NaN
2,Q03,Bagaimana tren Indeks Pembangunan Manusia (IPM...,llama3.2-finetuned,1,ok,3.287,7.959,35.503,46.757360,38.792969,NaN
3,Q04,Berapa Indeks Perilaku Anti Korupsi (IPAK) nas...,llama3.2-finetuned,1,ok,3.439,5.165,33.794,42.411815,38.863281,NaN
4,Q01,Berapa persentase penduduk lanjut usia di Indo...,llama3.2-finetuned,2,ok,0.002,7.385,35.871,43.272592,37.015625,NaN
5,Q02,Apa yang dimaksud dengan Indeks Khusus Penanga...,llama3.2-finetuned,2,ok,0.001,19.786,37.827,57.638229,36.707031,NaN
6,Q03,Bagaimana tren Indeks Pembangunan Manusia (IPM...,llama3.2-finetuned,2,ok,0.001,7.100,43.907,51.016972,37.496094,NaN
7,Q04,Berapa Indeks Perilaku Anti Korupsi (IPAK) nas...,llama3.2-finetuned,2,ok,0.001,4.171,21.748,25.935563,37.496094,NaN
8,Q01,Berapa persentase penduduk lanjut usia di Indo...,llama3.2-finetuned,3,ok,0.001,3.929,37.760,41.705789,37.328125,NaN
9,Q02,Apa yang dimaksud dengan Indeks Khusus Penanga...,llama3.2-finetuned,3,ok,1.934,15.317,47.374,64.644644,37.347656,NaN


,model_key,scope,use_rag,status,answer_status,cache_expansion,cache_search_rerank,n,question_count,client_median_s,...,backend_median_s,retrieval_median_s,retrieval_p95_s,citation_median_s,generation_median_s,generation_p95_s,peak_gpu_mb,peak_ram_mb,peak_system_ram_used_mb,peak_cpu_percent
0,gemma2-finetuned,production,True,ok,unverified,False,False,2,2,20.604613,...,20.5945,2.788,3.10480,15.0115,2.7935,3.95855,NaN,29.882812,15421.218750,100.0
1,gemma2-finetuned,production,True,ok,unverified,False,True,1,1,23.613975,...,23.5900,0.610,0.61000,17.5740,5.4040,5.40400,NaN,29.746094,14838.218750,100.0
2,gemma2-finetuned,production,True,ok,unverified,True,False,1,1,24.127129,...,24.1010,1.672,1.67200,17.2820,5.1450,5.14500,NaN,30.261719,15840.078125,100.0
3,gemma2-finetuned,production,True,ok,unverified,True,True,8,4,14.190608,...,14.1735,0.001,0.00165,12.8105,1.4705,4.96170,NaN,31.398438,15835.242188,100.0
4,llama3.2-finetuned,production,True,ok,basic_checks_passed,False,True,1,1,64.644644,...,64.6270,1.934,1.93400,47.3740,15.3170,15.31700,NaN,37.347656,16131.160156,100.0
5,llama3.2-finetuned,production,True,ok,unverified,False,False,2,2,44.584588,...,44.5750,3.363,3.43140,34.6485,6.5620,7.81930,NaN,38.863281,15910.117188,100.0
6,llama3.2-finetuned,production,True,ok,unverified,True,True,9,4,43.864569,...,43.8330,0.001,0.00200,37.7600,7.1000,19.60840,NaN,38.988281,16145.058594,100.0


llama3.2-finetuned generator q1/4 model_unloaded r1: ok
llama3.2-finetuned generator q1/4 warm r1: ok
llama3.2-finetuned generator q1/4 warm r2: ok
llama3.2-finetuned generator q1/4 warm r3: ok
llama3.2-finetuned generator q1/4 warm r4: ok
llama3.2-finetuned generator q1/4 warm r5: ok
llama3.2-finetuned generator q1/4 warm r6: ok
llama3.2-finetuned generator q1/4 warm r7: ok
llama3.2-finetuned generator q1/4 warm r8: ok
llama3.2-finetuned generator q1/4 warm r9: ok
llama3.2-finetuned generator q1/4 warm r10: ok
llama3.2-finetuned generator q2/4 model_unloaded r1: ok
llama3.2-finetuned generator q2/4 warm r1: ok
llama3.2-finetuned generator q2/4 warm r2: ok
llama3.2-finetuned generator q2/4 warm r3: ok
llama3.2-finetuned generator q2/4 warm r4: ok
llama3.2-finetuned generator q2/4 warm r5: ok
llama3.2-finetuned generator q2/4 warm r6: ok
llama3.2-finetuned generator q2/4 warm r7: ok
llama3.2-finetuned generator q2/4 warm r8: ok
llama3.2-finetuned generator q2/4 warm r9: ok
llama3.2-fine

,model_key,phase,n,ttft_approx_median_s,ttft_approx_p95_s,tokens_per_s_median,tokens_per_s_p95,ollama_total_median_s,client_total_median_s,peak_gpu_mb,peak_ram_mb,peak_system_ram_used_mb,max_system_ram_delta_mb,peak_cpu_percent
0,gemma2-finetuned,model_unloaded,4,6.431148,11.703795,52.417440,54.941733,12.095640,14.139253,NaN,315.800781,15073.910156,1048.765625,100.0
1,gemma2-finetuned,warm,40,0.134401,0.160224,48.915163,52.733044,1.114631,3.159741,NaN,32.113281,15332.378906,296.203125,100.0
2,llama3.2-finetuned,model_unloaded,4,7.057839,8.635511,31.989634,35.871297,10.098160,12.133579,NaN,315.250000,15035.812500,1349.546875,100.0
3,llama3.2-finetuned,warm,40,0.073412,0.121215,28.537548,33.219219,3.705905,5.762409,NaN,31.843750,14287.464844,879.328125,100.0


Cakupan lengkap: 112/112 pengukuran; 0 gagal. Gagal tetap tercatat, bukan dianggap berhasil.
CSV utama: D:\Politeknik Statistika STIS\Pembelajaran\Skripsi\SourceCode\SkripsiNabil\Hasil\tujuan3\20260927-214705-5142d6fe\system_performance_all.csv
Jumlah baris: 112 | Pertanyaan: 4
